In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.capstone_footfall")

DataFrame[]

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.capstone_footfall.raw_data")

DataFrame[]

In [0]:
display(spark.sql("SHOW VOLUMES IN workspace.capstone_footfall"))

database,volume_name
capstone_footfall,raw_data


In [0]:
MY_ID = "pragyachaturvedi"
VOL = "/Volumes/workspace/capstone_footfall/raw_data"

In [0]:
dbutils.fs.ls(VOL)

[FileInfo(path='dbfs:/Volumes/workspace/capstone_footfall/raw_data/export/', name='export/', size=0, modificationTime=1790696156968),
 FileInfo(path='dbfs:/Volumes/workspace/capstone_footfall/raw_data/raw/', name='raw/', size=0, modificationTime=1790696156968)]

In [0]:
import pyspark.sql.functions as F
def W(d, n): d.write.mode('overwrite').option('header', True).csv(f'{VOL}/raw/{n}')
S = "if(hour between 18 and 20, 1.8, if(hour between 14 and 16, 0.6, 1.0))"
K = "if(dayofweek(trade_date) in (1, 7), 1.6, 1.0)"
C = "if(dayofweek(trade_date) in (1,7), .85, 1) * if(hour between 14 and 16, .45, 1)"
st = spark.range(12).selectExpr("int(id) as s", "format_string('ST%02d', id + 1) as store_id",
    "format_string('CITY%d', id%6) as city", "if(id % 2 = 0, 'Mall', 'High Street') as format",
    "round(0.08 + rand(5) * 0.20, 3) as conv_base")
W(st.select('store_id', 'city', 'format'), 'stores')
gr = (st.crossJoin(spark.range(90).withColumnRenamed('id', 'd'))
  .crossJoin(spark.range(10, 22).withColumnRenamed('id', 'hour'))
  .selectExpr("*", "date_add(date'2025-04-01', int(d)) as trade_date"))
gr = gr.selectExpr("*", "(s * 90 + int(d)) * 12 + int(hour) - 10 as rn",
    f"int(round(30 * ({S}) * ({K}) * (0.7 + rand(6) * 0.6))) as fc")
gr = gr.selectExpr("*", f"int(round(fc * conv_base * ({C}))) as nb")
OFF = "pmod(rn, 54) = 0"
ERR = "(pmod(rn, 54) = 1 or pmod(rn, 216) = 2)"
W(gr.selectExpr("store_id", "trade_date", "hour",
    f"if({OFF}, 'OFFLINE', 'OK') as counter_status",
    f"case when {ERR} then 'ERR' when {OFF} then '0' else string(fc) end as footfall"), 'footfall')
bl = gr.filter('nb > 0').withColumn('m', F.explode(F.sequence(F.lit(1), F.col('nb'))))
bl = bl.selectExpr("format_string('B%02d%03d%02d%02d', s, int(d), int(hour), m) as bill_id",
    "store_id", "s * 90 + int(d) as k", "m", "hour",
    "format_string('%s %02d:%02d:00', string(trade_date), int(hour), m % 60) as bill_ts",
    "int(1 + rand(7) * 8) as items", "round(exp(6.0 + randn(8) * 0.7), 2) as bill_amount")
B = ['bill_id', 'store_id', 'bill_ts', 'items', 'bill_amount']
dup = bl.filter("hour = 12 and m = 1 and pmod(k, 8) = 3 and k < 1040")
ex = spark.range(135).selectExpr("format_string('B9%06d', id) as bill_id",
    "if(id < 75, 'ST13', format_string('ST%02d', id % 12 + 1)) as store_id",
    "format_string('2025-05-%02d %02d:15:00', int(id) % 28 + 1,"
    " if(id < 75, 11 + int(id) % 9, 3 + int(id) % 3)) as bill_ts",
    "int(1 + pmod(id, 8)) as items", "round(400 + pmod(id, 900), 2) as bill_amount")
W(bl.select(*B).unionByName(dup.select(*B)).unionByName(ex.select(*B)), 'bills')

In [0]:
display(dbutils.fs.ls(f"{VOL}/raw/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/capstone_footfall/raw_data/raw/bills/,bills/,0,1790696182369
dbfs:/Volumes/workspace/capstone_footfall/raw_data/raw/footfall/,footfall/,0,1790696182369
dbfs:/Volumes/workspace/capstone_footfall/raw_data/raw/stores/,stores/,0,1790696182369


In [0]:
def to_bronze(name):
    df = (spark.read.option("header", True).csv(f"{VOL}/raw/{name}")
          .withColumn("_source_file", F.col("_metadata.file_path"))
          .withColumn("_ingested_at", F.current_timestamp()))
    df.write.mode("overwrite").saveAsTable(f"bronze_{name}_{MY_ID}")
    return df

bronze_stores   = to_bronze("stores")
bronze_footfall = to_bronze("footfall")
bronze_bills    = to_bronze("bills")

In [0]:
print("footfall rows:", spark.table(f"bronze_footfall_{MY_ID}").count())
print("stores rows:", spark.table(f"bronze_stores_{MY_ID}").count())
print("bills rows:", spark.table(f"bronze_bills_{MY_ID}").count())

footfall rows: 12960
stores rows: 12
bills rows: 86859


In [0]:
from pyspark.sql.window import Window

silver_footfall = (bronze_footfall
    .withColumn("footfall", F.expr("try_cast(footfall as int)"))
    .withColumn("footfall", F.when(F.col("counter_status")=="OFFLINE", None).otherwise(F.col("footfall")))
    .withColumn("trade_date", F.expr("try_cast(trade_date as date)"))
    .withColumn("sensor_ok", F.col("footfall").isNotNull()))

w = Window.partitionBy("bill_id").orderBy(F.col("_ingested_at").desc())
silver_bills = (bronze_bills
    .withColumn("bill_amount", F.expr("try_cast(bill_amount as double)"))
    .withColumn("rn", F.row_number().over(w)).filter("rn=1").drop("rn")
    .join(bronze_stores.select("store_id"), "store_id", "inner")
    .withColumn("hr", F.hour("bill_ts"))
    .filter((F.col("hr") >= 10) & (F.col("hr") <= 21)))

silver_footfall.write.mode("overwrite").saveAsTable(f"silver_footfall_{MY_ID}")
silver_bills.write.mode("overwrite").saveAsTable(f"silver_bills_{MY_ID}")

In [0]:
print("bronze bills:", bronze_bills.count())
print("silver bills:", spark.table(f"silver_bills_{MY_ID}").count())
print("difference:", bronze_bills.count() - spark.table(f"silver_bills_{MY_ID}").count())

bronze bills: 86859
silver bills: 86594
difference: 265


In [0]:
gold = (silver_footfall.join(
        silver_bills.groupBy("store_id", F.to_date("bill_ts").alias("trade_date"), F.hour("bill_ts").alias("hour"))
                    .agg(F.count("*").alias("bills"), F.sum("bill_amount").alias("revenue")),
        ["store_id", "trade_date", "hour"], "left")
    .join(bronze_stores.select("store_id","city","format"), "store_id", "left")
    .withColumn("bills", F.coalesce("bills", F.lit(0)))
    .withColumn("revenue", F.coalesce("revenue", F.lit(0.0)))
    .withColumn("conversion_rate", F.when(F.col("sensor_ok"), F.col("bills")/F.col("footfall")))
)
gold.write.mode("overwrite").saveAsTable(f"gold_store_hour_{MY_ID}")

In [0]:
print("gold rows:", gold.count())
print("sensor offline rows:", gold.filter("sensor_ok = false").count())

gold rows: 12960
sensor offline rows: 540


In [0]:
(gold.coalesce(1).write.mode("overwrite").option("header", True)
   .csv(f"{VOL}/export/gold_store_hour"))

In [0]:
display(dbutils.fs.ls(f"{VOL}/export/gold_store_hour"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/capstone_footfall/raw_data/export/gold_store_hour/_SUCCESS,_SUCCESS,0,1790696234000
dbfs:/Volumes/workspace/capstone_footfall/raw_data/export/gold_store_hour/_committed_1806850872844011416,_committed_1806850872844011416,212,1790696234000
dbfs:/Volumes/workspace/capstone_footfall/raw_data/export/gold_store_hour/_committed_4108917673467091366,_committed_4108917673467091366,113,1790675767000
dbfs:/Volumes/workspace/capstone_footfall/raw_data/export/gold_store_hour/_committed_vacuum3948606876357034936,_committed_vacuum3948606876357034936,96,1790696234000
dbfs:/Volumes/workspace/capstone_footfall/raw_data/export/gold_store_hour/_started_1806850872844011416,_started_1806850872844011416,0,1790696233000
dbfs:/Volumes/workspace/capstone_footfall/raw_data/export/gold_store_hour/part-00000-tid-1806850872844011416-03198d38-0a12-4caf-a858-a1942c532c33-388-1-c000.csv,part-00000-tid-1806850872844011416-03198d38-0a12-4caf-a858-a1942c532c33-388-1-c000.csv,3251410,1790696234000


In [0]:
print(gold.columns)

['store_id', 'trade_date', 'hour', 'counter_status', 'footfall', '_source_file', '_ingested_at', 'sensor_ok', 'bills', 'revenue', 'city', 'format', 'conversion_rate']
